# HARVEY plus CATA plus UPGP on Kaggle

Self-contained CIFAR-10 / ResNet18 / BadNets 10% poisoning experiment. This notebook embeds the training code and trigger; it does not need GitHub access. Select a **GPU accelerator** in Kaggle settings and enable **Internet** to download CIFAR-10, or attach an existing CIFAR-10 cache.

The full profile runs 20 warmup epochs, up to 20 reference cycles at 14 epochs per cycle, four CATA adaptation epochs per cycle, 100 final classifier epochs, then five UPGP postprocessing epochs. It uses one GPU only, even if Kaggle exposes two. There is no AMP, reduced dataset, shortened full schedule, or automatic batch-size change.

CATA replaces reference-model benign-set unlearning. UPGP's gradient-projection component postprocesses the final model using HARVEY-estimated sets; its separate perturbation detector is not used. This is an exploratory hybrid, not a reproduction of all three papers or a guaranteed accuracy gain. If HARVEY finds no suspected poison, UPGP is explicitly skipped.

GPU execution has not been tested on this Mac. CPU execution and notebook checks are recorded beside the notebook. Synthetic smoke metrics are execution checks only.


In [ ]:
# Edit this cell before Run All.
PROFILE = "full"                # "full" or "smoke" (tiny synthetic data)
PROTOCOL = "release"            # "release" (14 ref epochs), "paper-text" (10)
SEED = 1
BATCH_SIZE = 128                # use 64 in a NEW run if GPU memory is insufficient
CPU_THREADS = 2
RUN_NAME = f"kaggle-{PROFILE}-{PROTOCOL}-b{BATCH_SIZE}"

# Optional: attach a previous notebook output and point to its COMPLETE seed folder.
# Example: /kaggle/input/<previous-output>/harvey-cata-upgp/runs/
#          kaggle-full-release-b128/harvey_cata_upgp/seed1
RESTORE_RUN_DIR = None

# Optional offline CIFAR-10 cache ROOT, containing cifar-10-batches-py/.
CIFAR10_ROOT = None

# True by default: a smoke check still needs a GPU in Kaggle.
# False is only for local CPU validation; a full CPU run would be very slow.
REQUIRE_CUDA = True

assert PROFILE in {"full", "smoke"}
assert PROTOCOL in {"release", "paper-text"}
assert isinstance(SEED, int) and SEED >= 0
assert isinstance(BATCH_SIZE, int) and BATCH_SIZE >= 2
assert isinstance(CPU_THREADS, int) and CPU_THREADS >= 1
assert RUN_NAME and "/" not in RUN_NAME and "\\" not in RUN_NAME and RUN_NAME not in {".", ".."}


## Check the environment

Keep Kaggle's preinstalled PyTorch/torchvision pair. This cell does not replace it. CUDA absence stops execution before training. If the import fails, restart with a compatible Kaggle image rather than mixing arbitrary torch and torchvision versions.


In [ ]:
import os
import sys
import platform
import json
from pathlib import Path
import torch
import torchvision
import numpy as np
from PIL import Image

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA runtime:", torch.version.cuda)
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a GPU accelerator in Kaggle settings, then restart and Run All.")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")
    print("Only GPU 0 will be used.")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
if CIFAR10_ROOT is not None:
    cache = Path(CIFAR10_ROOT)
    if not (cache / "cifar-10-batches-py" / "data_batch_1").is_file():
        raise FileNotFoundError("CIFAR10_ROOT must contain cifar-10-batches-py/data_batch_1")
    os.environ["HARVEY_CIFAR10_ROOT"] = str(cache)
else:
    os.environ.pop("HARVEY_CIFAR10_ROOT", None)


## Extract the embedded code

Code goes to `/kaggle/working/harvey-cata-upgp`. Every embedded file is verified by SHA-256. Running this cell again keeps existing data and run outputs. A source mismatch stops execution so a running experiment is not silently changed.


In [ ]:
import base64
import hashlib
import io
import zipfile

CODE_DIR = Path("/kaggle/working/harvey-cata-upgp")
CODE_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_SHA256 = 'fd0237e9335b414b5eedf41acfa80d1b65e55cc9fcc51b9ab19901fcbf366ec6'
FILE_HASHES = {'cata.py': '7496e4b32038dad87b8f93a66a7cfc30605ee5b84e5fbcf535dd2227e9e71522', 'experiment.py': '3eda61a217288b32d2d86b6aa529a880e3e24100afe9f53771607d46d42c824e', 'tests/test_methods.py': 'b9acd91d8c300f69d0aac7aa64fbdc034770ca312f23ed66eca562cb8d1968ad', 'upgp.py': 'a1c57fe9ea7f92350aefd313a38e5ced5d104bfe89d0e50d6039c66734873727', 'vendor/cifar_badnets.png': '5609678121bb40f46c1986d86b888099c8129f980fbf11cf868494930764af6a', 'vendor/models/target_models.py': 'a68631031e8ff1c1939b92f06e4e41db8c2ad16ba6e316abe2ee47f35e5f2e68'}
SOURCE_ARCHIVE = (
    'UEsDBBQAAAAIAAAAIQAMoLxDiQkAAIsdAAAHAAAAY2F0YS5wea1ZbXPbNhL+rl+B8xeRCU1b7V2vdcLOuK6myVwuzSRuOh2PhwORkMQzCbIAaFv15L/fLl5I'
    'kKJttXf6kEjAvuPZxS58dHR0cX55TtjvLVVFzSX59njxTUTuCrUllBN235RFVijy5vzj5+VvpKkLWfPjklHBC74htMF9zUpWVLJ4NvvMMlULSWh5R3eStJKR'
    'mjNSVFWr6KpkIDbb1iImH2pZqOKWnXC2ofiFyLYCPp4TXvM/mKhnWd1yBUsC99ZrsIRxRaQCcqmKTJJ1LYjaMrJuy5JsYbEWYE5JbrURpGJVLXbx7OjoaLYW'
    'dUWyutmBKU0tFMkZa/D3zP6uqNrO3A/gzraGR391TJzPZrOcrYnktJHbWgVVnbMyPJsR+AimWsHJA6cVOyO3tGxZnDNFs20QxlnT4r8lRCMINbn7oBfIEhkW'
    'UnCipcboKUvzIlPAWShWySD8Yg1QdXOTVlTeBMbZiNyAQ6nAw7DmgNs/lfUK4lHRDS9UmzOIlmBgd5mTYPl7TL4LX3lLqmAm2uAILTjLY4wciirWcCgmRnEh'
    '1wVIY4GnkKALQHBKXnt2kNcJWZx1vgpaABo+o49LIWoRzDX4PPqqlYqsdASC04gsruN56PQbN2PeVqwMQpIkoMxq1UfU22Uow5iWQPises5YDl4j6FjVqF1E'
    'jBiiILhOqzVDA5IkEIf7YBGZcGSsKIORcS/8wzCsfZQTJ5Su4DzjG7XVxz6WcWy1vSSLMNYU0geZhfgLEvjSyPdJrykEqGQllZJc1HwNiarOb5mQ7N86LTqM'
    'LO9pBllVbCDfS5CFyYxZmBfyP3UBFlSM8gjOJCvbHLMec/P4tsYQAWA0RrQshGWaYvDSNJCsXEcg9Q/mHQEuxo1N/BTyHYJhDg9lykCTD6lddTiMupPtTmpM'
    'H5Fc7RqWWMhw9fVXjyn86yIE1FMmoXIBRvvQ0Dy3UbEA7eNiD7NDRlc0VB3k7LbIWDKH+jEfql6XNR0o77NEbmnDyN+S/YjbrT+VOU9kj7V8S7FyI0bqqoHw'
    'YaH3FHGbUgbGLpumEfGyiwJgt2rSquDB6VOg2GOA5BwzjHABLNZX8j051WE+GBAe7+tneTskANfCg8JmI1Aq04DwgYBZNXFqUAX2HO+YOkJM1Enuk6koePFd'
    '9GZ3Gnxhg3ifTIVlWpitVSY+d1smAGLoIQQ9GpodDcA2+uzxv0b+gaWRn6ZpWdwYyjD0iiDg9Reuexcmzh6rWPrm9S/T5DT+GgoZtBUMvv4zIqyps61M/j5h'
    'cimSBTv+BzRQrNhsVZqzjO6Aa+Ed8f/lKn0iIw+5Tj07Cllw6DN4xgLjGFZ6pa2Y2FrVdan3zAKYeJhRltwZRLvDR2Vsw8SEZcMIlULrLQVG4vQgpUDrFHbq'
    '8GIzEp/VqI9cK9XfEHQHqTXUTjMUv67B/RPKfQBpG/yFg00ZMB1ukc5v06djBRi2uqPiCH0r3nNX3LSxkNZ9/4qbedpQAf8r6DygPwFnG10VC2hS0o2g+fU4'
    'EL3cg3x0wogCAq5HjF7jI16l3YWrF+3PSQfNDAGUUz1UsC/T9W8jMW5kSnVP/1xU+xyOzILBVEIGG7g25HNpalQK+2WAgoQ4Itz3t7wmzsbDq4heAeujC8L0'
    'cPLYaT9yEcDAGFz1dFf8ejwl6c4G/sdGFpqI4HgRPnE/aOAh6nrkXIe9NyAE9LJ0eASPOAfN7Ec42lum58pGQPNVtxJacSFsa7xioI+5iVgLlZqYs3tlJ2Sw'
    'RHWzE35MhMCrPPWGuglkwJEpmGFVcila5jmBk/ZueEXhwG2YkvcwuwydwOFu8Q1Z7ZCbtqV65Sx2M325g6Ea53dInQI6FDvb6yxiuRvnncgfYP7GEw/6MeAH'
    'qrLt+1pU3kR+opsBIAwhZBUM5nqChuAYcU7aJc7stdgwdVxzMITmtFH2GUFgOkEwIc1ItqV8w0x4S6xX2vW5JCtjTuy73H1HXS61bRWD0tKHCy42ghGDcUcy'
    'b/2Z00LC0fE4Bg8eiV824r7R64nr9RrgoWcDt6TfW0xy8FpXxXH73b8RdInzVJ0920sXLK6wbe58/miR7T3iquAt29t0U1GnzRW8PUqwTFGg7INzZVw/sxF4'
    'aYRdx7cFuwt6gTrn9+X1BDBJpYEWj913v25GJS9EsR6XwojQstnSpC+l+9KdTYkxqs+81rSMw9wz4E0RIkwMM+8dUptXKU2EtaDPA4NH27yCqYCmCqQyoIz0'
    'bawT3Uu8Jb5JkG6OANfhCN3YtWrxOawoS8OmNbB7TEXITw+Uqu7kmUp1B8SmE85dyTJQgsQgGwERyKEStIBKu/v2R/My1EqWT2cdAOx/aAK6WfXwi1+L928j'
    'Uy3yVwQbmoKWMKvjuw67c2c4bDMPvpJtSUq6J8Npguny7g58sm5YRl1z/QrRqKIC60X3+qBX4vOcVr8Gg9gEjQ65jreV9kyow+HcAiNL1y0Mppa91sErdwIa'
    'RmGqHefxBYxccsmVgNC8g6+eJyYjkQwux2DCwNAmbcfB7mGOLJkc1EdoUWkJ2SZxGUt3vwPOp+i8wLMPvC5oooDmFApSwYfZO1EqedMqbJDoCm4IPHfguzq9'
    '7l9joKLotYW/tl9S3DHqqdSUdSgGqapTfGwcYaE7EONlF2MbtcBYFTqz9hlt+zx60kFxE9cBfvbz6n33UqOtyFuBhURnm72usZj1t7WfTr798YpmN3dU5BPX'
    'Qh8VqVgzQdAhAGqx8TXGZzf/Tcd9BrjQirtGEp/Lg8GQ5eQOg6FfeRPvOX1iTDgejgnmQMJo3KsP7Ru0AXkeoKIhhf0zB2iHIdg+IunLJ7U7hqdnwn7l7BER'
    'fbJMzxvDhdFQpJvL/bZyQtJeS/8wd3fT/Gzgs1uOyNxFPpWMcSBzv/dfUObosXMfKO03kNEXDadncuyaENnDVSMF2HvYuL+miJanMJbQ1N4U4G8wddVHGr41'
    'xxlLh6N7Rb+gZQmXqWln3R/MoFP9pOiGHX9FesFEAjZfGX5owcUtNLZwee2y0j6mW7xqghh0B3PGN9CUz0PXuXov6kh05QiucU71n7mGl8X+25b1Jv7Xcvkh'
    '/Xh++fZn99Tldj5dnL9bDqNqH8AcxfLDzxdvPuE82S29+zi6TNzGr8u3P725TH9cXpz/ZqCEDmD2jhyJXds12XBpzkZg2uhe4KxrkJIH9+1LRAa4Sx4GP2F7'
    '7nk1t0hLHnwEfjl56HEHHCMoJQ+jhS/zGAytYHZ98UI7NsDjumzl1qv6NoM04ey/UEsDBBQAAAAIAAAAIQCst00vbhkAAF1TAAANAAAAZXhwZXJpbWVudC5w'
    'ebU8a3PbRpLf9SuwcV0BcCCIVGQnUcKt8/q1qXMcl+xs1ZVOhRoBQxErEMBiAEm0z//9+jGDGYAgJSV7rLJFADM93T39ngaf/OWoU83RZV4eyfLGqzftqiq/'
    'O/jmm2/+S1xdFdJ7+furF17TlaVsvGXVeH9/cfaP1//tfeu9fPHpBfz5/cPbDz95SqylV0jRlHl55dVN1VZpVXhCeUWVisKTd7Vs8rUsWxUD7IN8XVdN64nm'
    'qhaNkgfLplp7aVVvPP0kk7LGa36SiVa2ML1/qq8NnJVQqyK/NJf/VFVpvleKQdSixSEGwge4NEMaUWbV2lypjTJfaQlzUXZrQA8oKmuG+OGXdwbaL2tx1SPT'
    'Vk264iH01Qwqy8GIuCzjZVembV6Vglj1xpkUd21eqBgIFWb+K/iuZBvRl3eVyGTjTLjJFQCiCTBImUkvf3nz4mw+2xponrdAvIJ9XSvE4NMBD7yRwJEmXleZ'
    'BCRa2CbZJnxlJp5J9V628x/01jl4omD8XpIwyCbyVClqtapaHtjVV7UZeNnlRZZcAr4qQpn5p0yBuq40cpQUlVIHB2e//fbJW9COBUmyzAuZJGHcSFUVNzII'
    'Y5AgkKuDX1+/eA/Dgll88uP8JPLg7w/Hx/T35Pmz8ODjp1f8+Pjk+xnePj757hn9fT5/Hh4cHGRy6Skps0QURYBfwtMDDz4sHjHe4dt0t6zj6Qe8fWtRdqJI'
    'Rs/ypX6cdpmIc5WIG5EX4rIAOngxC4GGOGAsWs4ylyK9hr1SOBrk6VKW6Wotmmug9I0oQLF2Ds1kK5t1XuaqzVMY/qnp5KNxdGEisu26K2LAs7pN2uV3xwMs'
    'dmIyMZ43I627pG2kDG5E0Um9eI8eYNbKUlXN4DFtmGy7pvToNtIp0hWICUDD/4uqBEIMKNCFUrWiTPUikZflabsN68v1qYNOSJbwOvJuvNysk7dyrYLw627I'
    'QdvVBfwtgOXh9hLtpjaUBuOl7DqMuUuhZpZoq3WeJmj6ArR1keeyBZCrgbv4IL7N21WiuuUyvwv8uF3XftiPiW8bIAQYe9cGCCrOwOopQ0FeZqBpi+MQ7L7/'
    'P6U7r5F1IYBWXKFXJnEjNS7AiRaZK2/y1OBE9879przyL1AztTqBqaEnQRg5SoYGqL/NAoC3YHJ/u2eo+3FEeTheo4JbxV9j3ABvsfB8HO17YOuk9x6kJXw4'
    'B2kxIrvfQVoNcYb59/ALTBpAkIhkMMUwEXmXkZfCXUDE5d7QTGn2iW07ZVh4OUB3wJa0V40MRNgrq5Z44AGIaT5Nmq0hzKynAwhNC6EUO0wVaJem6YPA4DdY'
    'qsvL9gfwETDCI4ZEXlUeZnKNOLRNfnUlm58AM2/ZFYWnHR4GD7nk4AJhIT+TBMxbmyRgNYtlxBDB04Atg70Fj1PlIOELEqhG5OWCbE8EmgQW45qvHDVFILGB'
    'QRcMCPbC3AVeCyWaRmwCfhYOZ/OKuHmyDdaiDoBSg0c4GstYwFj+MnzYe2003PHLCvypkkFw/ik+o71+2VR18B04v1pkGXjSxQkQaR6eVbArEAIE89kOpdn6'
    'mKl/r5r8c1W2onhT5HUQXpA5RuaxupxfgGl4CEjA9FP1ic03ofYeiBFF/lkG6McjD7x1eDFmH7BCNsg/ZDSxmeQormpZBhQmHHk+xy5Hab4UDQQXWQkKEdeg'
    'JmD8q/JGNm3gn739mz9m+Fqoa9oaZ6W/LLzZgSNPhSy1OG3bb3gWOGIRuvPA9KB/sKIIhvTOAcHCvnBl7JzGXPRDCCrKWtm6y+hhlpZ8OZAfcB40Ah2II4On'
    'g00yy9PfGMPuINwecN6z6WLEJ+fJYJrBeTbm1VCI9S5ifMi7SsuFoVZVbR0LiniDzETBeQYadynaFN3LCoxxIQc6q5eywXIwDqzjj90lamIPEj2z1UtYIEQk'
    'aI1EgWwueLkdAm6Q0H8jzBiS26q5lo1aQMRZ52Wyluuq2Wg8NWFrcS0TxCGAOFvZMAevYgiLMeIle6vW1bV0DC5YWNYFbeABmOgKsrwEKrbRIn7uYDA8ioE4'
    'CSZUBYDT8bPnEJj8AN/QWtA/oDhDE78AwGSKLYSNUT1RXkmYFXr/4c3t5t61O1c4eegK7WCJE3cJtDCW+CeQgYgMEksMYIERKUR5UoGnKDax917eell1W6LM'
    'QGJT3IqNQo+5QTVoV9IrYcCyKkAq4h4ixDwgyxBtN1WFaFQqhpQ4byAKAv0NfM58E51RJWhvfIt5CnuQY16KDiE4p2xlAJGN5XARazS9c2O/UBL8yMNLndrg'
    'XV78CB8eaQz8XYI4OTX59cPH0fR7BvpWmfHSsKXE2DCoKS6tkZ8O5UAgPAFQZH0P57ND0hipDusNwUxYmeZgjGnnIGIOGY+edMtSdiwLk8NCTMTqz+w0XhsT'
    'l6jf7YWD6mInYAi07oWro4FHAb6LvE0EehCxIBMgMjWRdVl8k3NqRYGharfG4D0zhKD3ocN96h7XkNZ12sOjT9qE4fkpeg2+8J568TzkrTUM1tEYY88rudxl'
    'DFRd5O30YB5QgTFtbmTm+udN6LoTM+Kcp130riGFlN9FhBio49eqTJgTDHdZiBbufZZNFQCT0UPzQHD1LoRzO/ECoQ2vdYxnqTNJGBIdMa0Ro4UOILNsMTRo'
    '4w2GhAsjlJqPAvYn3kesi2EUmmNwQ5tCigM5L+RnTS7KVlFsvcyxCKQLLswTCKQTBocm5Zy9F98w2UvMvh0Azi5sZjwaM5XXnLMAYCKjJ2mhAuSuSaL0ygsX'
    'ja38f7rKgR+iBNA2ZaIAnSHF/gBzPhswncbGbRXYLIHSxw5E+SZXAJtGRB5rCTl+FHjDbdiUus3X4KAbeKBSzFHdSEDDx60NTOYDISz4+0opyOMwiqaKED3D'
    '7UHhRtPmxBzOurGNCCKqmzh8oZkL784hB+64xJmRPc4xynJy1QjkYZu0VQKiKh3ZxE9RXeUtygEL2537BHmKpZM4bZAgIL9BneMppKaNzDoqNC58hO0PwkXm'
    '2DAi7IEexrM5mAx9/dTTUG1ZBRiyFnfBPEQR21jAQ2/N2vDu7QtMKjvY0l4LGrmU4IBSWz7G8DgtOsxbvFsBxqyOp3HT+Wt+BYaO7x16gG7Y4zvgEUzgu/Ea'
    'tDoYsADTXFNVWqK6SoIYDgkAAQLVOetKrAu/bhrIXXxIjnmCSwhMjf3hDlHRC6jJgikJUK2snQcknt63C49N9gAQySs8WoJstoRlvxdEuJ3R0wXAnKBxgojX'
    '67rdOASQqiD7FQXIhhaT6Bgsjhg2qOt/6oq2FuOQyxlggwaK6+rsQDXljSg0+f3magu1Uxs5QERy8RqjiJFqOpsHQfSluMzBpufS0SFHK8NYVcuWBTkGM7Wu'
    'g7k8/D7y5rFl/xPv9Z1IW+ADiCrK8NnL1+9QsgLQqcMV8podFQGA523lAZST0MovDFxRFPUmhq8JfA82A1MBpjHmrXXwOBniocVY1JD2ZsFhMCTvqV4kBk0N'
    'kLBuTURR+ZMOMYJwsJ/gVCExTiGAK4UWfKmMEYboEFJXiiXovimfaO5WDeyIcfhXkM23/Xx8fClL0M5tx613+a8L7yR+Pnt27MY4SVqBcHos/BoYKDZeMTgd'
    'jsAW41bClmFwEz8Dwl0AYUSpcjx7prVCg9K0s2TjKpjzERnnpwhTp9IEQzHml2CP8FIXdc55LkQR67wEyFftqvdoWFBfk0MJtieOJmAkBtYqFopKv4Bt2It8'
    'n7MzZojY6YUVaJNYD2oArkFj9M/p+YX3s8FL3xiaNa3kWpy4NDEYMAQGtmc+Eh+O9RgOJsy0TbvMgkjTrhHp5vG2Ia2aRqYUWJOBNNHjv8lGUOQ6ZRp6F8dK'
    'ZFM+jc+3XJUJGAR6QVK6cL9B1+ybz2YgCAaUNalUGZBtk6fKcMoNRyf4ZY4qfBqXGD77p1ss14AGMIbJpC9UMzFxe+mvfXXmKmg6ACpvJBZPnj6lMwNjJ5rq'
    'Fpj7xUenA3DN+S2Ixm1AGgAXn+lkBjxwhUUhtICR5xM4mDEC+9WGr7gspmKaWTEeWxSQX1JR0IfsDA82IcWTYu0Uc+maTzvcgw7Ac3i4UTe4taMREXjfTq10'
    'iKaL913p1m9sbR0D33YFy2WK0zR9Eer6ILKxD2Z06E6h+unM78tAZhhE8aJrK59yhp3HdHsqsBT9+yDHe2DTGOeRPRKYPChBVGwAtffUcCr4oAaDruzHx95H'
    '8K+gC8J7++F3FEG4bERLVtHTXQmwnrysqmuspGP5p4+3RlgiXjArIIojjfEWQv9AmdLo6AV0xwPkIHhUrbgNAil9CTiBDGLBA8/b9bL9Ae2o7gZgkixHD2kq'
    'BnBH+fCFBuLjElNEfa1TQrhc+gjkSw/uq+/Ci9fX8H/ApRulSyBUUEkqN6vN8bwubzEjwRPMm9G5ZcMSGZrDS+TeNabUfiPBgkn/6wDKua8P3TOdENK5HVVP'
    'XDnpR5NA0BgWjSS5kQ02HyTJaKSqOogWE2zjkIpmfEGweEwGUUN+IyErogJ/GJ6aZo9YrcTxs+c0SGTJ5aaVWE2KV/Iuy68kePV9eqA/fRELQxdINKki1lwV'
    '1WXgP43rjR8SU/SuaWmqsWbWahsEZkdQl8ai32w2RnSXrBFvXbqS6TVEKBTbOEPt/bhufSPEPQBTJnMOCziuZwGifbo3TVn6cMkCok69L3r1r7H3O4w8PGQo'
    'WEYQAHkFcg/3uvIQRTMeJotkCamsGvQoEv/pyBi4BdJj9vX+7OmMFzZKfJSCmznSRVdsFvKyfLnExiNsHcFz1cwyZohX4HAUySlaZn04wUD8sF1f+i8KxH4D'
    'DnhdF7KVDnf8oZkf0EKudrwjzk5OLzqdQPZrOwB+gu9VhWaYStXo4tzNGCbX7tG/YU/U70JoS317ClsJVTecEt7oVIJrcTUfbju8trPjEtTFBTcI6lFpAepY'
    'd+3sttqpwaaWZGttvVm0J8s48In3jrrNLB8VqAnmsx1k2DC4hDCioRhxMIY7k8priEbQEqAt0kU4OsDuDRhODOxEiOdFnWCDG5dW2Mfcyvxq1aoETyJ0Gap3'
    'tVrR0I9MCAs73i/9vvo1sAvDJZ/LIOjAZF2BTT3FepWfbmAf9XewkzWkpHgV4yOsS5/S0X3fkeIT73yydkiWFSCOtJE6PrFPsAkm0B0GPMs5YdzdpmBYhqYU'
    'mYYR3roC1ajKPA1MQAVjq04lGmPbymBIuLAnp6AZ67x11WiIFE7W7WUcoobjgT1Q7N0YL/3tFoqQaGr8LSBq53A2fUSzYa9ekPfsguIjvW0W+6pue2Gi6k/8'
    'IhNrRh2dCqh4i+dmePTZLOIZpIeukfH7itHULuoFdu2jnevsJfo/EimEx+mSYRwJ2oVOmZiShG6Oy2K6uLZdrtXGxmZhdMOkYYPq7dC8EgpxV2OaENCaC0by'
    'W2/uFHsXSK1DaBiO8LoylipyVGgLHuK/oHLfKOtl2bObt13rcEpcmthBcmTnrkSx1If3sSlIaG4MJdZQTWK08PuaXI84KDvp/QIw54x7oasXuMbFjjNBTsn1'
    'SBx4ehFZfSB3sJi5lS7d5tvI/vBCKjo5tVVCNssoQXh/mTcKoiOID/mUg5CMhybmITbccqKu6sCR2shpxhrszx4FtOyzIivLq7yUxuzw0W46PMbDc1oe5bRA'
    'nU6IKKGY8oHtELvbFZ7aa5TYUF94Py9Ym+haDQHSPWsM9ZTBEEujVk6kcUbe5D50LeGxMd99ly03aY3MJ37+jLXS+N5vsfRCj7Ja+Hmo5epFYNp44ec+A6bh'
    'srYZwHvMl5WPP2vCGDvXjLkGge0A/f9ws0ayNjZt+HnivXBzFMgCICDFXq+8ZM3nYO2QIkkMxEmjLyVshHSatuMR0F9KkJGmqzEksKM86n/c6NCrXQk3eua6'
    'Dh53cch/9v5tPBYsMqC8L2zbQD62d9bEXSavRpuwEhDhbvztwbRt98iAWWtCBvqjx+10AT/b52EOklZ9p1W3B2Js16DH3qDKAr5gE4PdClripykFglDqtVXQ'
    'W2NA6TLqPuK3T0GnFxrKrzaWA9F9+pSQGXtvfRB0j4+dKGVH9txh+/jCpDhbpkob4IU33+a/ZgD5uMSq4L74c5t0wn5Mu3bMzulG79ZJxLmmPo5MsI3ximJn'
    'ZyLibnzgwLOP3MjANOm1hoxb8J+tCMHFcndthTm9MAzHHV+wlA1oZyulg5qp+O8e/2+2rWcGOIR+D3e6WWc3+0CByzX+/uhPLzcRZNDk7VUuGyls2y7aSsME'
    'oGlf0DKCp3PkuqqK7YM0PZ15jRHG4EANP3tzuy2BdjIDevRoLWxkWsF+kGw6mJ+74M6dBxf2EPDCAUL9WQ+b//PW9DV3NY3ONc+3rFiP67B9FiMolHQE84AC'
    'zq+wXcyTVr/7lnWpxJK4MUNKYH1H/QSRiqqxsj2Mo8FKDNoFdJXls1tmIfhxWX/2ex1dc0GnI5AyS3Q7O/LODeTfNlVHffMd+lTdfIG1Ce7A6LKc0BZLcNOE'
    'GPd5wchMpnkms9iR4ZaWYt5ik2ej4MY8C3DRQeknRrfQT/xXBwkEV6E1+/mM1j+1jEYDOaJlMIjomrQ6PuOFdWWwVym93oZnTAbbIzvfRA580b/hcR9YgCqK'
    'YhumQ/A0iJ5YCKPWeUmZEYAZcY93csw9vQYx5+tkrW8sH4nmM1c+I8P2QROR44/IzqD33Rr3xHsDHFq5TWmmnojltVE/m6C2o76d7c/kfPqcgjX1+oKPK/gA'
    'xxRyovCr979OkYz43BfKCONxnQxZCF+ZzVgdRpnCP5atD0kmGfa+Ys7Ht692Zkfg69YV1rS79SL+0ZQJE1AysVlQl4hlBYTC2Qh00SR0uytkE//aFW3+sZX1'
    'u7OAIs81JJuQ1ZVSR3/2+v+5ftSju2tWj/QfqToRx6fztu2ePivkww4id9FRl4ApiuuaIorJzuB2Omt9SOuei9hEm95Ex8GwY3Ab0oM71sxnf+caq3nf9DVu'
    'X3Pxn2pjc/kxamQb7olpMZlqXZsSqQlgfPqPxxMPbI3YE48bWzGdOTOSYKmpW9A0DzmNDf+WimXk9QqyMETvKgZshaTO64ajLCRyDxg5RU/wzWs8YdxfcMRR'
    'bq3Rmbx4KM8fWchzBllr8UeqfYT74Jx0R/KfUHZMw6l65hQTyDuMSwlPvFeDt6UvMaQCE82BnVctdbn0UGIji8AAQYd+vVLRkeUwmcjU/e38ID8/zmazYVf/'
    'wF6F56c0GqlJdKS5VTb06SX7/gB7RxnOBMhC6UN45wX96cLAVEt02FdFdqeIUx9LhX5nJHJuyRJyvI17RxSw9Q8sbT3KU9sliibEvYdA3njuI3LXOljWv1gA'
    'O/KHCpNM166apNv8hkj87T3sBp6FKzDXdzL7yUOvk2P7B4QylPFAmnNZOAdX5lPSq5UJ7xx1Qd+3hdrl7YKUtwQH/wQD4NszqL+CX12xbndYVWKNe7jbxU/b'
    'bHYXyWzTKbYkODhP16fkXSpBPj62Vf1Li41G+fj1yD/Hgj+I1mOjC/zoCGP0Cxp9854p6ug3WaY8pPk8OsLAz/4oA38oZmdsYbDfF1/gB+UJGyHwzWbQWxQq'
    '1hS8CXmjo8y78cyXPDtG1u4/M3E/dg6905QE2gAE9kE0sqF0uoSojQMc89kbM7nBysAvO6HF06eP98rmMz5M8nq495tVWTzIvY5PisckJeo6h6AQ0jqwYgLr'
    'F9gQY90oy2rvapUjOtzlgxnjw3kQDaIh/9RWtu3NySNUX1OJPbEO0Vi0kNR14WS2vn5n19zVl5NgzS8lOWPpOuo7a7BjBp46V5OAdHqcKJlWZaZwxmO6HSZh'
    'qk3ZriQEPCNKnJeRI7d/DRYv2CFiKl4I1bJ/822e/cCQlRM/7aDdkHVH+cPt+Iq0ZOwqe5huKyp9mKH9yw5KJkiqMR/4kx/mR6riF81Vh3HAB7zCpkeVNnlN'
    'vT9JklVpkugWl1hkGYKh0YF/eGikJ8K2Lnqt7tycQ0UDrRldGokEkv7V5Y3MHOO/vYqRO3cV/I0MBMrbdYGaQGHmgp/sAcVy6cLS770guFpAQHqIPX8DmGbE'
    'LrCkLBH91M2CemnMzPmuGbrR1EWDmpR1L+/pjL7V3QANGrELIoUYhxhiTGNy/MOumbpxe3Laya5Jpo/S32bT4d4t0C24YMH0m3t8bN+CAOycgzJzyDHl47BE'
    'Ods38dneiUVjJlHWbKfZnoAdM41Zn1xzfrx3rk4T/gihlE3swvnH7/dOpsRjx9z5TL+mhe3A1Cbc2xOTt9IPA4i+8zukyAW7Zu09asS1V1T+jFHO49jtXa9j'
    'yTGWFTJv3akWAj18BUz/vgGFS7Y5Xjghtvezd8xrUar5szfjKy3ncGM+sRrN59ev/rrwjiPqfcevM+CJnglX895m2cUhgQbSnaNpEG43D+ovtVj013qrwx0o'
    '0ewjnnTE+On3tww/6krl2EduccGYbwbwhJtgYofOHJmAT7FKbyJf4SadzCV7w3BuxyRMI+2MosFFZhNU5CW9V8CB8gqkq7HhrMZbDBvxkD/j/ha8xwef+M2t'
    'oIJIHsMeBfMTFoT+Vxq5U4otNx+K4C8D0eD5bKYXtiVlLK0+h2f4qyE/zi6sZO37rZIR5t5iAnW6ybjT1xHycweYI0M01BGi0UhH3BdaAN0k0/ntkTGN+me1'
    '+K0qASFCjj/lg+qUJERjkmDenSSaTHwDyNX48OD/AFBLAwQUAAAACAAAACEA9BqRDA0EAACOCwAAFQAAAHRlc3RzL3Rlc3RfbWV0aG9kcy5weaVWTW/jNhC9'
    '+1cMUBShAIWwHe8iaGGgaVqgh+weWu8pCAhGGstsKFJLUk6yv75DfVlWnE3aCpCjjIZvZt48cajKyroAtVEhoA8z1f4frMt2s62zJWQySOjM19ZstcrC1R6d'
    'x09YWvecknP1IErpH1K4vtpcfTEapTPoUvBGVn5nQ4tUV0XVI91Lr7yIZuHso0+hcvZvzEIK97XSuaD3SNa6xVKmENp63+I0yfE6KO15PsruN3q+sTKPkTdo'
    'vHXR4jHMZrNMS+/hE4adzT3ry+Ub+rkml+SnGdCV4xaiXXhVGIGaElLWCGly8Q2dFUGhZx71tvOPV9mwAOuT5LCLZOLIZZ6ztoLQ5MhulzyFBd3nS36XvMP/'
    'vHFuflZHK3o3H4gwTgWjCyIj4pD1YEXhsJABWZLCyyzmEfOiwTzmo9D2XmrRdPoBsfInqfhO/EEjk1q6ulOIN1/cUVp8+SFJB8zvXMdI8zHS/EQNncLahhq1'
    'U5p4IA3WWkcNCl/JDKclRXFSb2+PYzVpz9s4KTShu6h3yd2wdk/RrCMVP6IqdiHiTGTPWu0jfSzF83rBD638Af6QLodDzqA8kMh3trBG6p/BoZZPmMPvX/ni'
    'EnKLHowNcUGpPAI+ySxAFC0fMAtKYFLHBWW9ovtDzPxQNPpaB/LuwrMiBTZU00JYQ9xp9YCsqy55nxBb7Kn4bucdoU0i7wLqczukQx9bCp/pKRknSdapEjru'
    'xoooZch2VBE+VfQNqxANTj2dVngpTU1fg0fM2Sp5RSuOdg3DVilcDOmMTCOVvKWP5EQDW6RlRBreUupUDEliTY6/QEcMPiMxAOewJ1trypUshq6Rdc83/4rx'
    'Rg1D1gmV14eeEh1nB7GMBLIncuOmLu7r7ZY2yGZT7Tf3t2ke76M2Rz0wYQz/C7/WaIKSmg22G2UIO3K9Gugn86+xzZ+tKxexc6MXB//lSH55Oz8o2NE8YeMm'
    'XI5DSLIVyC4T+JGADq0xBeHHGTGej6yphNirbLbz60UK2q35fHFYFwmjVf0YbRdMYXk3JHu4wxhkXf40VGPdNNW+4Xo1qm9rHRhZYgp7qWsEZVp2eTTmfavY'
    'qDNvSKSBSZu8byPG3YtkSQy0HofWCx/iOJpU9mpijbfIacyyhKuA5f/M7liv8YQiKPYj7b2k1uwhPrwtzuV/FufHsQb/xJsv7IQoPx6L8imF58lG8LoGj6TU'
    '7DCH49UJwRzLPEZKJuJJ4Syr6rPOin49Kj4e0SjE5NDWh+nh+r8NzvFiPnB+sEf2uzZuXI0di8pvFR3ikMVlJxVdEQ8lBnRTVQ8vXkjnjVD9/jcA8MLJvNWT'
    '5wURFgPQHORSaxanzkxtQYhoFQLWazgTNFeUEeKsDTycQ6OVav4HUEsDBBQAAAAIAAAAIQDTHsvAaAQAAG8JAAAHAAAAdXBncC5weZVW227cNhB9368YbB5M'
    '1Vp2tygQxO0G6cVJ+mIYjWOgMAyBK40k1hLJkNTamyL/3qFI7SV+SCvYeyGHM2cOz8zsfD7/eP3uGi4/OQ6rl4vVqwv47fojfLj9HXQNzovyASvoxA4tNFZU'
    'EpUHqx8dn83e//Ln7eVf4AZjOokO0HnZC0/2Dr0DqZxHUQU/IcaZA4PWD3YjvNQKKvRYem357Nrqv+ljWBwc+XkHb4D9AQu4hUqKhnWi31Qig1t+k+UgtlpW'
    'UjXkQDkEEw9rS4jm8/lM9kZbD7RQtidfuFK8HtQYR3QgHLydzWYV1rARTrqitrovQmYsvOSACm2zWy/5q5cUtDOtWK+WPLuYAT2UppVPsE6uS+HTqUr262XG'
    '607TUsZLM7BsPFHk4Aj10Ambw7bdH+0koWm421YsOs2hHrquGL+U6NZvRecw+vDaE/L13hF3nwZhkeK4oU9xZJ3Mfl7DChc/LCPg8Fgk9lWKi73xuxSSu1YY'
    'vFvd57Akho8NltGrFeqBAvdSsQ4VmwCQsVSexQMOBb05Ihwr9hxiOfQBZQgQqYXvItIsg3NYxTiPKJuWtLMGNlI+7pDh5O7uIiC5h++n/edb54eV5X1CHxPf'
    'tsmG3/BSKy+bQQ+OEaAUNukhSYpNes+jQtLVE8HTBld03SPRoG00Avq70gqfsT6dGde3UbH5cb5b7vU+JK9wS7efH0JVfmcwg5oCbYn0BGn0VnfCk4e9qcXx'
    'Pg/O4vUu6XoXqxNG2Hh0ASx+eDMBC5QnaNlhlcpv8l0It3efTWU0yK4qCBc61usKuxyoCiokvU/ZUOmXLWn6x2+U1wu43JKEgxeoLeJn6gq/XoV25CU1mdLB'
    'Yys7pGqra7SEQdIGtYSyQ6EoqqNmMJZpgMGRfKXqCEVKVP2jLuDufiRT5WACn9FUiR6rwghL7x4tiSPct4kX/RpWXyLhdE6qCqlWGf3vsuAA1dCjpfbHYtbZ'
    'QQLkYjSH1+uJgsNmeDYWxcN+JUL5jFYXgWNG3bTwulAkq/WNHVIzCM9bXlpKtiAGrDa7SDt7CkqKlGck7t3xV76hjv4obMUOXv4LDad4KSEVrEc6haqIoYA0'
    'iF9p/1UB7AuBjO/UPRfGoKpYPEJCpxnTTq1yr11zKlpeduSTZdnRtX6LoACSwIiuG3szJxUMJM3sKBcrJI2Q27Bxaa227OxK7+vIjcSM49HsJ9RPR1Nug0o2'
    'Kgy7kPjYL/nZSX0FnT2bLpP2k+yziX87Mcqlx56AfkmFNSiStVUk8CJIeyquGJ7uTUun1VRkKbsgSxJ6tIkrhpYMrUX7cU1p248T5f8LaTzf6UaO3SvZm2cW'
    'L+CDFxsq1QXZstXCFF7YBj25K3VvhsCjb60emjZ4owmBTwb0ln5uaN/Sa9kJRx0l1bNw4efIOgWeVHG0xx3NYpJswVYh3SM8dxf5KMwgqDiwpKrj0VATlaZb'
    'oDacpvIEhcVIeRhDC/h6M8YMmxnvqfWwk9tP7J7v/c/+BVBLAwQUAAAACAAAACEAts3EomYAAABqAAAAGAAAAHZlbmRvci9jaWZhcl9iYWRuZXRzLnBuZ+sM'
    '8HPn5ZLiYmBg4PX0cAkC0gogzMEEJP9IvF0MpAw9XRxDKuYkJ0iwNnPl/Iw0uL2ir01sD1fRzdpPZry6EqyLE0R6mgINuNwOakiwOkswMMQFsnuopsapAjUz'
    'eLr6uaxzSmgCAFBLAwQUAAAACAAAACEA9O21XfwDAABRFQAAHgAAAHZlbmRvci9tb2RlbHMvdGFyZ2V0X21vZGVscy5wee1YTW+kOBC996/wKodAhkXNR1qj'
    'kThMVsppNxrtzC1qIQeqEytgWBt6Mvvrt2zTDQa6O7uTyc4hRGrAfn5VdtUrm5yfny8IXn+CvIGGME4+fftSiezBX5xj14KVdSUa0qgm68XnnFBJOB+3+puW'
    'Zw2rOC0U4HqxyAoqJbmikmVXRZU9Ogj6o8rbAtwP2jg81ZRLHEISEix0Uw4bkqaMsyZNHQnFxkPf0rqgHKRHdnfZCJZDEnQ86pJtDcLpjSEIR7v+nsztodjh'
    'ZxXfBmgXffoNH8PcmbHzCIJDkUr2NyTR3qy5IYrmOeP3SeCRO0Zlck0LCSMzd7wzckWb7OGmEiVaMvQzDoWWQ8/zJniuI+FhR2yofMC4Zm1j8J/hrxZ4w2gx'
    'WEK26cyTXzB0pBJ9lFSLptlH98J09LE6bcmCqms2TLNm7HUKJlEbrJA3Z2a4PrMGXGuU2+ftphJfqci7tH0aJGelp3jtCyhaZ5cXTp+HzpPruiP0Lmo9LHSw'
    'Z4R7l9griUzzVtXQfY+AphVcAaYqDV9VpuGbTt90+tPpNBdVjc/OCcViRJOljyFuBMXsxch+ES1MBbijOyjp5xCh0r9X6jutV01TAIcTO3L8MlLfG3txpf+/'
    'ev531eWIqCfeRLPePE9ER1yIpi4c0M1bmenN/CzHARs9LSEHjg9RD4x+yPGhqynmO2JST2aKx505nfO2TPWjNM+aBmQSLLsqmObQgChxrGxYZkLkkQ17gjwt'
    'qxwjWlNBSwVKJUCe3FQcJuXH+HWk9GDqHua0c9R86ZSUt7TQ3c7hkWMR9XpIyCpenKp8WGZW8XefJGZK4CoeoQr6DUSwS5i0pI+Q6janC5Tyow/W7XI9KPYz'
    'VOERqiB8b3EFPVc4xxUd4QovVxZXeIIrPsJ1GYQWV3SYi27v66oqzMJ+zGndsC183N5/wkZcXgcDErjjmq5Sm20YCDPsd8aBCgetkgsjh76oWGLoec66meiR'
    'oUUTLJfL0aheeoO5Wurb1cahCs2MhwrSDSpjb83jmrwjGLQLpx/26yAJtB0NX+/bsAjutgbGd4y2rMwwn9Y18NyExLE1Mz5muO507xgKrHuYLO64gNmbzIVx'
    '5MdU8L3CjN7seoovodWva3pgASILoGt5aAFiCxArQDS7KXQprACxBdggAm/+lsFX1euruuMs8XQ6jPKZwhR7YyYJ1ehZY33ujzD9HkLOPG3dI7dq1vol1L+R'
    '/o3XfUTuoUmhvANd/dKclXrNXfJhzDsyrtJjAxS7QOKepajMzhC8dwbSUR+4/3X/wbGDHahzo9sWh/+Wug09/be2RDtr9Jg515pGFL/6NCIv9lZe9KLTuFy+'
    'yjQGnyQvNo1/AFBLAQIUAxQAAAAIAAAAIQAMoLxDiQkAAIsdAAAHAAAAAAAAAAAAAACAAQAAAABjYXRhLnB5UEsBAhQDFAAAAAgAAAAhAKy3TS9uGQAAXVMA'
    'AA0AAAAAAAAAAAAAAIABrgkAAGV4cGVyaW1lbnQucHlQSwECFAMUAAAACAAAACEA9BqRDA0EAACOCwAAFQAAAAAAAAAAAAAAgAFHIwAAdGVzdHMvdGVzdF9t'
    'ZXRob2RzLnB5UEsBAhQDFAAAAAgAAAAhANMey8BoBAAAbwkAAAcAAAAAAAAAAAAAAIABhycAAHVwZ3AucHlQSwECFAMUAAAACAAAACEAts3EomYAAABqAAAA'
    'GAAAAAAAAAAAAAAAgAEULAAAdmVuZG9yL2NpZmFyX2JhZG5ldHMucG5nUEsBAhQDFAAAAAgAAAAhAPTttV38AwAAURUAAB4AAAAAAAAAAAAAAIABsCwAAHZl'
    'bmRvci9tb2RlbHMvdGFyZ2V0X21vZGVscy5weVBLBQYAAAAABgAGAHoBAADoMAAAAAA='
)
raw = base64.b64decode(SOURCE_ARCHIVE)
assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256, "Embedded archive checksum failed"
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    assert set(archive.namelist()) == set(FILE_HASHES)
    for name in archive.namelist():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe embedded path")
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == FILE_HASHES[name]
        target = CODE_DIR / relative
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError(f"Existing code differs: {target}. Use a new clean working directory.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
print(f"Verified {len(FILE_HASHES)} source files in {CODE_DIR}")


## Validate methods and optionally restore a run

The CPU correctness tests check CATA aggregation, buffers and UPGP projection. Restore only your own trusted checkpoints: they include Python objects. Set `RESTORE_RUN_DIR` to the previous complete `seed1` folder, including both `metadata.json` and `checkpoint.pt`. The runner rejects changes to seed, profile, schedule, device, code or PyTorch version.

A checkpoint is written after each warmup/reference/final epoch and each UPGP epoch. Interrupted CATA unlearning replays from the last pre-request epoch checkpoint. Completed runs are skipped. Saving local checkpoints does **not** guarantee survival after a Kaggle session is discarded: retain the working outputs as a saved notebook version or downloadable output and attach them to the next session.


In [ ]:
import shutil
import subprocess

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=CODE_DIR, check=True)
RUN_DIR = CODE_DIR / "runs" / RUN_NAME / "harvey_cata_upgp" / f"seed{SEED}"
if RESTORE_RUN_DIR is not None:
    source_run = Path(RESTORE_RUN_DIR)
    for required in ("metadata.json", "checkpoint.pt"):
        if not (source_run / required).is_file():
            raise FileNotFoundError(f"Missing restore file: {source_run / required}")
    if not (RUN_DIR / "metadata.json").exists():
        if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
            raise RuntimeError("Destination is nonempty without metadata. Use a different RUN_NAME.")
        RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_run, RUN_DIR, dirs_exist_ok=True)
        print("Restored previous run into", RUN_DIR)
    else:
        print("Existing writable run found; keeping it instead of overwriting with the restore copy.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
environment = {"python": sys.version, "platform": platform.platform(),
               "torch": torch.__version__, "torchvision": torchvision.__version__,
               "cuda_runtime": torch.version.cuda, "device": DEVICE,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               "profile": PROFILE, "protocol": PROTOCOL, "seed": SEED}
(CODE_DIR / "notebook_environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print("Run directory:", RUN_DIR)
print("Synthetic smoke data" if PROFILE == "smoke" else "Full CIFAR-10 training protocol")


## Train HARVEY plus CATA plus UPGP

Run this cell once and let it finish. Output streams to the notebook and `training.log`. If you interrupt the cell, the child process is stopped; rerun to continue from its last complete checkpoint. Do not start a second copy against the same run folder.

The release profile follows the local runner's settings, including its documented differences from the paper. Low reference-model clean accuracy is intentional; final clean accuracy is measured during Stage 4. UPGP can increase or decrease accuracy/ASR; use the saved before-UPGP metrics to isolate that postprocessing step. Changing hardware introduces another source of variation when comparing against Mac runs.


In [ ]:
import signal

command = [sys.executable, "-u", str(CODE_DIR / "experiment.py"),
           "--variant", "harvey_cata_upgp", "--profile", PROFILE,
           "--protocol", PROTOCOL, "--seed", str(SEED),
           "--batch-size", str(BATCH_SIZE), "--threads", str(CPU_THREADS),
           "--device", DEVICE, "--run-name", RUN_NAME, "--resume"]
print("Starting:", " ".join(command))
process = None
try:
    with (RUN_DIR / "training.log").open("a", buffering=1) as log_stream:
        process = subprocess.Popen(command, cwd=CODE_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   start_new_session=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_stream.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training exited with status {return_code}. Inspect training.log; rerun to resume.")
except BaseException:
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    raise


## Results and downloadable artifacts

`result.json` records final clean ACC, ASR, before-UPGP metrics, seed and elapsed time. `split_quality.json` uses true poison IDs only for reporting, never for training or splitting. `metrics.jsonl` includes all phase logs and explicitly records a skipped UPGP step if the estimated poisoned set is empty.

This cell works independently after a kernel restart, searches working files and attached outputs, and never starts training. Download ZIPs from the Kaggle file browser; inline FileLink URLs may return 404.

The compact export below includes the final and pre-UPGP models, metrics and split IDs. It excludes `checkpoint.pt` to avoid duplicating the large resume artifact; **preserve the full working output directory separately for resumption**, including its checkpoint, metadata, and embedded source. Neither the compact export nor a saved cell output alone is sufficient to resume.


In [ ]:
"""Read/export completed results without training or prior notebook variables."""
import json
from pathlib import Path
import re
import zipfile


def recover_results(search_roots=None, output_root=None):
    roots = [Path(p) for p in search_roots] if search_roots is not None else [Path('/kaggle/working'), Path('/kaggle/input')]
    output = Path(output_root) if output_root is not None else Path('/kaggle/working/harvey-results')
    discovered = {}
    for root in roots:
        if not root.is_dir():
            continue
        for file in sorted(root.rglob('result.json')):
            if 'harvey_cata_upgp' not in file.parts:
                continue
            try:
                result = json.loads(file.read_text())
            except (OSError, ValueError):
                continue
            if result.get('variant') == 'harvey_cata_upgp':
                discovered[str(file.resolve())] = (file, result)
    if not discovered:
        print('No completed HARVEY+CATA+UPGP result.json found in the current files.')
        print('This does not prove the training failed. The earlier session may have ended or the run may be in a saved notebook version.')
        print('Open the completed notebook VERSION and inspect its Output files. Attach those outputs here if needed.')
        print('If no files were preserved, inspect the original executed notebook output for its printed FINAL CIFAR-10 RESULT.')
        checkpoints = [p for root in roots if root.is_dir() for p in root.rglob('checkpoint.pt') if 'harvey_cata_upgp' in p.parts]
        for path in checkpoints:
            print('Checkpoint found (not proof of completion):', path)
        print('This recovery cell does not start or resume training.')
        return []
    output.mkdir(parents=True, exist_ok=True)
    exports = []
    for index, (file, result) in enumerate(discovered.values(), start=1):
        folder = file.parent
        label = 'SYNTHETIC CHECK ONLY' if result.get('synthetic') else 'COMPLETED RESULT'
        print(f'\n{label}: {file}')
        print(json.dumps(result, indent=2))
        before = result.get('before_upgp')
        if before:
            for key in ('clean_accuracy', 'asr'):
                if key in before and key in result:
                    print(f'UPGP change in {key}: {result[key] - before[key]:+.2f} percentage points')
        stem = re.sub(r'[^A-Za-z0-9_.-]', '_', f'{folder.parent.parent.name}-{folder.name}')
        export = output / f'{index:02d}-{stem}-results.zip'
        with zipfile.ZipFile(export, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for name in ('metadata.json', 'result.json', 'metrics.jsonl', 'training.log',
                         'split_quality.json', 'split.npz', 'poison_ids.npy',
                         'before_upgp.pt', 'final_model.pt'):
                path = folder / name
                if path.is_file():
                    archive.write(path, arcname=f'{folder.name}/{name}')
            environment = next((parent / 'notebook_environment.json' for parent in folder.parents
                                if (parent / 'notebook_environment.json').is_file()), None)
            if environment is not None:
                archive.write(environment, arcname='notebook_environment.json')
        print('Export created:', export)
        exports.append(export)
    print('\nDownload ZIPs from the Kaggle file browser: /kaggle/working/harvey-results/')
    print('If viewing a saved notebook version, use its Output files instead of inline links.')
    print('ZIPs exclude checkpoint.pt; preserve the original complete run folder separately for resume.')
    return exports


if __name__ == '__main__':
    recover_results()


## Protocol and interpretation notes

- Final training is supervised CE as in the released CIFAR config, not MixMatch. SGD uses LR 0.1, momentum 0.9, effective weight decay 1e-4 and milestones 60/80/90 for 100 full-profile epochs.
- CATA keeps the notebook/local hybrid's fixed anchor, top-30% magnitude mask, sign aggregation, scale 0.7, AdamW LR 1e-5, weight decay 0.1 and four adaptation epochs per reference cycle. These are starting values, not tuned performance claims.
- UPGP uses CPU layer-wise SVD, 97% gradient energy and relaxation alpha 10, then five SGD epochs at LR 0.001 without momentum or decay. BatchNorm buffers are frozen during its postprocessing. Full epoch and candidate settings are in `metadata.json`.
- The final checkpoint is reported; no best-test-epoch selection occurs. Compare clean ACC and ASR together, and use multiple paired seeds. A single Mac-vs-Kaggle accuracy difference is not evidence that a method improved.
- Full runs can outlast a session. Session limits and GPU availability vary; this notebook does not assume a guaranteed runtime or persist files outside Kaggle's saved outputs.

References: [official HARVEY project](https://intellisec.de/research/harvey/), the supplied CATA and UPGP papers and existing project implementation, [Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks).
